# Olfactory Projection Fibers (OP_1)

In [1]:
from ip.ios import *
from ip.enhancement import *
from ip.binary import *
from ip.graph_nx import *
from ip.swc import *
from ip.utils import *
from skimage.morphology import skeletonize
from skimage.util import img_as_ubyte, img_as_float
from skimage.filters import frangi, hessian, meijering
from scipy import ndimage
from ip.spherical import iterate_spherical


## Read Data

In [2]:
folder_path = "./OlfactoryProjectionFibers/ImageStacks/OP_1"

In [3]:
images = load_tif_stack(folder_path)

In [5]:
single_download(projection2d(images), "./Test/images/projectionOP_1.png")


True

In [ ]:
imgFloat = img_as_float(images)

In [ ]:
img_cropped = images[:25, 314:445, :75]

In [ ]:
# Iterate through volume in spherical coordinates
for r, theta, phi, value, (z,y,x) in iterate_spherical(img_cropped):
    if value > 253: # Example condition
        print(f"Found bright point at r={r:.4f}, θ={theta:.4f}, φ={phi:.4f}")
        print(f"Cartesian coordinates: z={z}, y={y}, x={x}")

In [ ]:
slide_imshow(images)

## Code

In [ ]:
#filtered = ndimage.gaussian_laplace(images, sigma=3)
#hessianM = img_as_ubyte(hessian(seg2, [0.75], black_ridges = False))

In [ ]:
filtered = ndimage.median_filter(imgFloat, size=3)
filtered = img_as_ubyte(filtered)

In [ ]:
threshold2 = mean_threshold(filtered)
binary2 = simple_binary(filtered, threshold2)
seg2 = segment(images, binary2)
# threshold2 = mean_threshold(seg2)
# foreground = simple_binary(seg2, mean_threshold(seg2))

In [ ]:
er = img_as_ubyte(ndimage.binary_erosion(binary2))
skel2 = img_as_ubyte(skeletonize(er))

### For visualization purposes only

In [ ]:
# simple_imshow([blended(skel2), blended(er)])
# slide_imshow(skel2)

In [ ]:
#download(hessianM, "./hessian")
#download(loG, "./loG3")

### Graph generation

In [ ]:
graph = Graph(skel2)
graph.set_root((3, 427, 33))
graph.create_graph()
root = graph.get_root()
g_root = (30.979,429.04,0)
print(f"OP_1 GOLD STANDARD ROOT: {g_root}\nTEST ROOT: {root}")

In [ ]:
mst = graph.apply_dfs_and_label_nodes()

In [ ]:
graph.save_to_swc(mst,"./Test/OP_1test.swc",1.0)